# Group 8 — ISIC skin-lesion classification

Coding component skeleton. Every requirement from the client document maps to one section below.
Target runtime: Google Colab, single TensorFlow / Keras stack.
Fill in the TODO blocks in order; each section must stay runnable from top to bottom.

**Datasets (all three are used)**
- ISIC 2018 Task 3 (HAM10000, 7 classes) — train / validation / internal test, lesion-disjoint split
- ISIC 2019 (8 classes) — external validation, SCC excluded, hash-deduped against 2018
- SIIM-ISIC 2020 (binary) — external domain-shift test, patient-level split

**Fixed label space (7 classes)**: MEL, NV, BCC, AKIEC, BKL, DF, VASC


## Requirement map

| Section | Requirement | Main artifact |
|---|---|---|
| 0 | setup | runtime, output folders |
| 1 | 2.1 | library version table |
| 2 | 2.2 | gradient tape demonstration |
| 3 | 2.3 | training vs evaluation memory |
| 4 | data foundation | `manifest.csv` |
| 5 | 2.4 | uint8 cache and `tf.data` pipeline |
| 6 | 2.5–2.6 | augmentation figure |
| 7 | 2.7–2.9 | `model.summary()`, feature maps |
| 8 | 1.1 | gradient-magnitude figure, VGG vs ResNet |
| 9 | 2.10–2.13 | training history and learning curves |
| 10 | 2.14–2.17 | confusion matrix, top-3 confident errors |
| 11 | 2.18–2.20 | external results, unseen image, calibration |
| 12 | submission | saved artifacts and results archive |

Coverage checklist: 1.1 · 2.1 · 2.2 · 2.3 · 2.4 · 2.5 · 2.6 · 2.7 · 2.8 · 2.9 · 2.10 · 2.11 ·
2.12 · 2.13 · 2.14 · 2.15 · 2.16 · 2.17 · 2.18 · 2.19 · 2.20


## Run plan (free Colab runtime)

- **Session 1**: sections 0–5, build the manifest and cache images to Drive.
- **Session 2**: sections 6–9, train and checkpoint weights every epoch.
- **Session 3**: sections 10–12, evaluation figures and packaging.

Every session starts by restoring the Drive state from section 0; never retrain just to regenerate results.


## 0. Setup and runtime

One runtime switch decides where data lives. `RUN_LOCAL` selects the local layout (`data/raw`, `data/cache`) or the Colab layout (`/content/data/raw`, Drive-backed cache); every later cell builds its paths from `DATA_ROOT` / `CACHE_ROOT` / `OUT_DIR` only, so no personal absolute path ever appears in the notebook.

Dataset folders downloaded once on your machine can be used as-is: `download_if_missing()` skips any dataset that already has JPEG files.

Produces: reproducible runtime, output folders, fixed seeds.

In [ ]:
# Section 0 - runtime setup. One switch decides where data lives:
#   RUN_LOCAL = True  -> data/raw and data/cache next to this notebook (development on your machine)
#   RUN_LOCAL = False -> /content/data/raw and Drive-backed cache (Google Colab, the delivery runtime)
import os, sys, json, hashlib, random, subprocess
from pathlib import Path

RUN_LOCAL = not Path("/content").exists()
DATA_ROOT  = Path("data/raw").resolve() if RUN_LOCAL else Path("/content/data/raw")
CACHE_ROOT = Path("data/cache").resolve() if RUN_LOCAL else Path("/content/drive/MyDrive/isic_cache")
OUT_DIR    = Path("outputs").resolve() if RUN_LOCAL else Path("/content/outputs")

# Mount Drive first; creating folders under /content/drive before the mount would break it on Colab.
if not RUN_LOCAL:
    from google.colab import drive
    drive.mount("/content/drive")

for _folder in (DATA_ROOT, CACHE_ROOT, OUT_DIR / "figures", OUT_DIR / "tables", OUT_DIR / "weights"):
    _folder.mkdir(parents=True, exist_ok=True)

SEED = 8

def set_seeds(seed=SEED):
    """Fix every random source used later (Python, NumPy, TensorFlow)."""
    import numpy as np
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    import tensorflow as tf
    tf.keras.utils.set_random_seed(seed)
    return seed

set_seeds()
print("RUN_LOCAL:", RUN_LOCAL)
print("DATA_ROOT:", DATA_ROOT)
print("CACHE_ROOT:", CACHE_ROOT)
print("OUT_DIR:", OUT_DIR)
# TODO(0): verify CACHE_ROOT is writable before caching images there.


## 2.1 Libraries

Required: list every library used and its version. Produces: printed version table saved to `env.json`.

In [ ]:
# TODO(2.1): import tensorflow, keras, numpy, pandas, scikit-learn, matplotlib, PIL, kaggle.
# TODO(2.1): record versions (python, tensorflow, keras, numpy, pandas, scikit-learn) to env.json.

## 2.2 Forward pass and automatic differentiation

Required: explain how the framework tracks the forward pass and recovers gradients. Produces: per-layer gradient norms for one real batch.

In [ ]:
# TODO(2.2): run one real training batch forward inside tf.GradientTape.
# TODO(2.2): call tape.gradient(loss, model.trainable_variables) and print per-layer gradient norms.
# TODO(2.2): write the markdown explanation of what the tape records and when gradients are released.

## 2.3 Training and evaluation memory

Required: explain the memory difference between training and evaluation mode. Produces: behaviour evidence and written comparison.

In [ ]:
# TODO(2.3): call the model with training=True and training=False on the same input and show the difference.
# TODO(2.3): show that evaluation needs no gradient tape (tf.stop_gradient / inference path).
# TODO(2.3): explain activation memory during training versus inference memory in the report.

## Data layer: one manifest for three datasets

Foundation for 2.4: download, hash dedupe, label mapping and group-disjoint splits. Produces: `manifest.csv` and the split counts printed below.

**Acquisition, choose either route at any time**
1. Download locally with the Kaggle CLI into `data/raw/<name>/` (no Drive needed). The notebook then skips downloading and uses those folders.
2. Let this notebook download on Colab; a Kaggle token (`~/.kaggle/kaggle.json`, or `KAGGLE_USERNAME` / `KAGGLE_KEY`) must exist once.

**Rules enforced below**: fixed 7-class label space, SCC dropped from ISIC 2019 (no model slot), byte-hash dedupe across datasets (ISIC 2019 largely repeats ISIC 2018), lesion-disjoint split for 2018, patient-level split for SIIM, deterministic SIIM subset, and one fixed unseen image reserved for 2.19.

Paths in `manifest.csv` are stored relative to `DATA_ROOT`, so the file survives a move between your machine and Colab.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# Fixed model label space (7 classes). This order never changes anywhere in the notebook.
CLASS_NAMES = ("MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC")
CLASS_INDEX = {name: i for i, name in enumerate(CLASS_NAMES)}
MALIGNANT_CLASSES = {"MEL", "BCC", "AKIEC"}
DX_TO_CLASS = {"mel": "MEL", "nv": "NV", "bcc": "BCC", "akiec": "AKIEC",
               "bkl": "BKL", "df": "DF", "vasc": "VASC"}

# Kaggle sources. Replace the ISIC 2019 slug with the one in your dataset page URL
# (https://www.kaggle.com/datasets/<owner>/<slug>). Any mirror that keeps the same
# image/label file names works, because the scanners below discover folders automatically.
KAGGLE_SOURCES = {
    "isic2018": {"kind": "dataset", "ref": "kmader/skin-cancer-mnist-ham10000"},
    "isic2019": {"kind": "dataset", "ref": "PASTE_ISIC2019_DATASET_SLUG"},
    "siim2020": {"kind": "competition", "ref": "siim-isic-melanoma-classification"},
}
ALLOW_DOWNLOAD = True   # set False when the folders under DATA_ROOT are already populated
SIIM_SUBSET_N = 6000    # 33k SIIM images are not needed for a domain-shift check


def kaggle_ready():
    """The Kaggle CLI reads ~/.kaggle/kaggle.json or KAGGLE_USERNAME / KAGGLE_KEY."""
    return ((Path.home() / ".kaggle" / "kaggle.json").is_file()
            or bool(os.environ.get("KAGGLE_USERNAME")))


def download_if_missing(name, root=DATA_ROOT):
    """Download one Kaggle source into root/<name> unless image files are already there."""
    dest = Path(root) / name
    if dest.is_dir() and any(dest.rglob("*.jpg")):
        print(f"[skip] {name}: already present ({dest})")
        return dest
    source = KAGGLE_SOURCES[name]
    if "PASTE_" in source["ref"]:
        raise ValueError(f"Set the Kaggle slug for {name!r} in KAGGLE_SOURCES first.")
    if not kaggle_ready():
        raise RuntimeError("Kaggle credentials missing: save ~/.kaggle/kaggle.json "
                           "or set KAGGLE_USERNAME / KAGGLE_KEY.")
    dest.mkdir(parents=True, exist_ok=True)
    if source["kind"] == "dataset":
        args = ["datasets", "download", "-d", source["ref"], "-p", str(dest), "--unzip"]
    else:
        args = ["competitions", "download", "-c", source["ref"], "-p", str(dest), "--unzip"]
    print("[download]", source["kind"], source["ref"])
    try:
        subprocess.run(["kaggle", *args], check=True)
    except FileNotFoundError:
        subprocess.run([sys.executable, "-m", "kaggle", *args], check=True)
    return dest


def _jpg_index(root):
    """Map image stem -> path for every JPEG below root, regardless of folder layout."""
    return {p.stem: p for p in Path(root).rglob("*.jpg")}


def scan_isic2018(root=DATA_ROOT):
    """HAM10000 / ISIC 2018 Task 3 -> image, filepath, label_7, lesion_id."""
    root = Path(root) / "isic2018"
    files = _jpg_index(root)
    metadata = sorted(root.rglob("HAM10000_metadata.csv"))
    rows = []
    if metadata:
        frame = pd.read_csv(metadata[0])
        for _, row in frame.iterrows():
            stem = str(row["image_id"]).strip()
            if stem not in files:
                continue
            rows.append({"image": stem, "filepath": str(files[stem]),
                         "label_7": DX_TO_CLASS[str(row["dx"]).strip().lower()],
                         "lesion_id": str(row.get("lesion_id", "")).strip() or stem})
    else:
        truth = sorted(root.rglob("ISIC2018_Task3_Training_GroundTruth.csv"))
        if not truth:
            raise FileNotFoundError("ISIC 2018: neither HAM10000_metadata.csv nor "
                                    "ISIC2018_Task3_Training_GroundTruth.csv was found.")
        frame = pd.read_csv(truth[0])
        image_col = frame.columns[0]
        onehot = [c for c in frame.columns if str(c).upper() in CLASS_INDEX]
        for _, row in frame.iterrows():
            stem = str(row[image_col]).strip()
            if stem not in files or not onehot:
                continue
            rows.append({"image": stem, "filepath": str(files[stem]),
                         "label_7": max(onehot, key=lambda c: float(row[c])).upper(),
                         "lesion_id": stem})
    if not rows:
        raise ValueError(f"ISIC 2018: no labelled JPEG matched under {root}")
    return pd.DataFrame(rows)


def scan_isic2019(root=DATA_ROOT):
    """ISIC 2019 -> image, filepath, label_7. SCC is dropped because the model has no SCC slot."""
    root = Path(root) / "isic2019"
    files = _jpg_index(root)
    truth = sorted(root.rglob("ISIC_2019_Training_GroundTruth.csv"))
    if not truth:
        raise FileNotFoundError(f"ISIC 2019: ISIC_2019_Training_GroundTruth.csv not found under {root}")
    frame = pd.read_csv(truth[0])
    image_col = frame.columns[0]
    onehot = [c for c in frame.columns
              if str(c).upper() in CLASS_INDEX or str(c).upper() == "SCC"]
    rows, dropped_scc, dropped_other = [], 0, 0
    for _, row in frame.iterrows():
        stem = str(row[image_col]).strip()
        if stem not in files:
            continue
        labels = [c for c in onehot if float(row[c]) > 0.5]
        label = str(labels[0]).upper() if labels else ""
        if label == "SCC":
            dropped_scc += 1
            continue
        if label not in CLASS_INDEX:
            dropped_other += 1
            continue
        rows.append({"image": stem, "filepath": str(files[stem]), "label_7": label})
    stats = {"rows": len(rows), "dropped_scc": dropped_scc, "dropped_other": dropped_other}
    return pd.DataFrame(rows), stats


def scan_siim(root=DATA_ROOT):
    """SIIM-ISIC 2020 -> binary label, patient_id. Public labels exist only in train.csv."""
    root = Path(root) / "siim2020"
    files = _jpg_index(root)
    metadata = sorted(root.rglob("train.csv"))
    if not metadata:
        raise FileNotFoundError(f"SIIM-ISIC 2020: train.csv not found under {root}")
    frame = pd.read_csv(metadata[0])
    rows = []
    for _, row in frame.iterrows():
        stem = str(row["image_name"]).strip()
        if stem not in files:
            continue
        rows.append({"image": stem, "filepath": str(files[stem]),
                     "label_binary": int(row["target"]),
                     "patient_id": str(row.get("patient_id", "")).strip()})
    if not rows:
        raise ValueError(f"SIIM-ISIC 2020: no labelled JPEG matched under {root}")
    return pd.DataFrame(rows)


In [ ]:
def md5_of(path):
    """Content hash used to detect the same image stored in more than one dataset."""
    digest = hashlib.md5()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def add_hashes(frame, column="file_hash"):
    return frame.assign(**{column: [md5_of(p) for p in frame["filepath"]]})


def assign_group_split(frame, group_col="lesion_id", fracs=(0.70, 0.15, 0.15), seed=SEED):
    """Split by lesion / patient so one group never appears in two splits."""
    groups = sorted(frame[group_col].astype(str).unique())
    np.random.RandomState(seed).shuffle(groups)
    n = len(groups)
    n_val = max(1, int(round(n * fracs[1]))) if n >= 3 else 0
    n_test = max(1, int(round(n * fracs[2]))) if n >= 3 else 0
    n_train = n - n_val - n_test
    if n_train < 1:
        n_train, n_val, n_test = n, 0, 0
    parts = np.concatenate([np.full(n_train, "train", dtype=object),
                            np.full(n_val, "validation", dtype=object),
                            np.full(n_test, "test", dtype=object)])
    return frame.assign(split=frame[group_col].astype(str).map(dict(zip(groups, parts))))


def stratified_subset(frame, n, label_col="label_binary", seed=SEED):
    """Deterministic class-balanced subset so the rare malignant class survives sampling."""
    if len(frame) <= n:
        return frame
    rng = np.random.RandomState(seed)
    kept = []
    for _, group in frame.groupby(label_col):
        take = max(1, int(round(n * len(group) / len(frame))))
        kept.append(group.loc[rng.choice(group.index.to_numpy(), size=min(take, len(group)), replace=False)])
    return pd.concat(kept).sort_index()


def absolute_paths(frame, root=DATA_ROOT):
    """Manifest stores paths relative to DATA_ROOT so the file moves between machines."""
    return frame["filepath"].map(lambda p: str(Path(root) / p))


def build_manifest(root=DATA_ROOT, cache_root=CACHE_ROOT, subset_n=SIIM_SUBSET_N, seed=SEED):
    """Scan three datasets, dedupe, label-map, split and write manifest.csv."""
    if ALLOW_DOWNLOAD:
        for name in KAGGLE_SOURCES:
            download_if_missing(name, root=root)

    isic2018 = scan_isic2018(root).assign(dataset="isic2018")
    isic2019, stats2019 = scan_isic2019(root)
    isic2019 = isic2019.assign(dataset="isic2019")
    siim = scan_siim(root).assign(dataset="siim2020")
    print(f"scanned -> isic2018={len(isic2018)} | isic2019={len(isic2019)} "
          f"(SCC dropped={stats2019['dropped_scc']}, unmapped={stats2019['dropped_other']}) "
          f"| siim2020={len(siim)}")

    pooled = pd.concat([isic2018, isic2019, siim], ignore_index=True, sort=False)
    pooled = add_hashes(pooled)
    before = len(pooled)
    pooled = pooled.drop_duplicates("file_hash", keep="first").reset_index(drop=True)
    print(f"duplicate images removed: {before - len(pooled)} "
          "(kept the first occurrence: isic2018 > isic2019 > siim2020)")

    # 7-class label where it exists; binary malignancy everywhere. External sets have no label_7.
    pooled["label_7"] = pooled["label_7"].fillna("").astype(str)
    if "label_binary" not in pooled:
        pooled["label_binary"] = np.nan
    pooled["label_binary"] = pooled["label_binary"].fillna(
        pooled["label_7"].map(lambda c: int(c in MALIGNANT_CLASSES))).astype(int)

    # 2018: lesion-disjoint 70/15/15. 2019: external validation. SIIM: external domain-shift test.
    split2018 = assign_group_split(pooled[pooled["dataset"] == "isic2018"], "lesion_id", seed=seed)
    pooled.loc[split2018.index, "split"] = split2018["split"]
    pooled.loc[pooled["dataset"] == "isic2019", "split"] = "external_val"
    pooled.loc[pooled["dataset"] == "siim2020", "split"] = "external_test"

    external = pooled[pooled["split"] == "external_test"]
    if len(external) > subset_n:
        keep = stratified_subset(external, subset_n, seed=seed)
        pooled = pooled.drop(index=external.index.difference(keep.index)).reset_index(drop=True)
        print(f"siim2020 external test subset: {len(keep)} of {len(external)} images")

    # One fixed image for requirement 2.19; it is never used for training or model selection.
    pooled["is_unseen_demo"] = False
    demo_pool = pooled[(pooled["split"] == "external_test") & (pooled["label_binary"] == 1)]
    if demo_pool.empty:
        demo_pool = pooled[pooled["split"] == "external_test"]
    demo_index = demo_pool.sort_values("image").index[0]
    pooled.loc[demo_index, "is_unseen_demo"] = True
    print("unseen-image demo:", pooled.loc[demo_index, ["image", "dataset", "label_binary"]].to_dict())

    pooled["filepath"] = pd.Series(
        [str(Path(p).resolve().relative_to(Path(root).resolve())) for p in pooled["filepath"]],
        index=pooled.index)
    columns = ["image", "dataset", "filepath", "label_7", "label_binary",
               "lesion_id", "patient_id", "split", "is_unseen_demo", "file_hash"]
    manifest = pooled.reindex(columns=columns)
    manifest[["lesion_id", "patient_id"]] = manifest[["lesion_id", "patient_id"]].fillna("")
    manifest = manifest.reset_index(drop=True)

    print("\nrows per split:")
    print(manifest["split"].value_counts().to_string())
    print("\nrows per dataset:")
    print(manifest["dataset"].value_counts().to_string())
    out = Path(cache_root) / "manifest.csv"
    out.parent.mkdir(parents=True, exist_ok=True)
    manifest.to_csv(out, index=False)
    print(f"\nwritten: {out}")
    return manifest


manifest = build_manifest()
display(manifest.head(10))
print("label_7 values:", sorted(manifest["label_7"].unique()))


## 2.4 Preprocessing and label encoding

Required: preprocessing, augmentation and label encoding. Produces: fixed class order, uint8 cache, `tf.data` pipeline, one-hot labels.

In [ ]:
# TODO(2.4): define CLASS_NAMES in fixed order (MEL, NV, BCC, AKIEC, BKL, DF, VASC) and the label maps.
# TODO(2.4): resize to the model input size and cache uint8 arrays on Drive to avoid re-decoding JPEGs.
# TODO(2.4): build tf.data datasets for train / validation / internal test with one-hot targets.

## 2.5–2.6 Augmentation

Required: show two random augmentations of one sample side by side; explain variance without label change. Produces: augmentation figure.

In [ ]:
# TODO(2.5): define the training-only augmentation pipeline as Keras layers.
# TODO(2.5): display the original image plus two independent random augmentations side by side.
# TODO(2.6): write how augmentation changes variance while labels stay unchanged.

## 2.7–2.9 Model, summary and feature maps

Required: build the model, print summary, extract feature maps, explain shapes and backpropagation. Produces: summary file, parameter counts, feature-map grid.

In [ ]:
# TODO(2.7): build ResNet-50 (ImageNet weights, include_top=False) + GAP + dropout + softmax head.
# TODO(2.7): print model.summary() into a text file and report total and trainable parameters.
# TODO(2.8): extract activations from one hidden layer for one test image and plot the channel grid.
# TODO(2.9): explain why spatial size shrinks and channel depth grows, and how backward updates weights.

## Part 1.1 support: gradient magnitude per layer, VGG vs ResNet

Required by 1.1: a graph of gradient magnitudes per layer for VGG vs ResNet. Produces: the Part 1 figure and its data table.

In [ ]:
# TODO(1.1): build a VGG and a ResNet with the same input size.
# TODO(1.1): run one forward and one backward pass, record the gradient norm of every layer.
# TODO(1.1): plot gradient magnitude against layer depth for both models and save the figure.

## 2.10–2.13 Training and learning curves

Required: adaptive optimizer, fixed epochs, loss curves with learning rate or gradient norm, overfitting analysis. Produces: training history and curves.

In [ ]:
# TODO(2.10): choose AdamW (adaptive optimizer) and a fixed number of epochs; state the learning-rate schedule.
# TODO(2.10): compute class weights from the training split only.
# TODO(2.11): log per-epoch training loss, validation loss, learning rate and gradient norm.
# TODO(2.12): plot train and validation loss with the learning rate on a second axis.
# TODO(2.13): mark where validation loss starts to rise and write the plateau and overfitting analysis.

## 2.14–2.17 Internal test evaluation

Required: test accuracy and loss, normalised confusion matrix, top-3 confident errors, error analysis. Produces: evaluation figures and tables.

In [ ]:
# TODO(2.14): evaluate the selected weights on the internal test split (accuracy and loss).
# TODO(2.15): plot the row-normalised 7x7 confusion matrix.
# TODO(2.16): show the three wrong predictions with the highest predicted-class probability.
# TODO(2.17): write the error analysis and check it refers to the current final image IDs.

## 2.18–2.20 External datasets, unseen image and calibration

Required: external evaluation, unseen-image probabilities, softmax discussion, calibration. Produces: external table, bar chart, reliability diagram and ECE.

In [ ]:
# TODO(2.18): evaluate on ISIC 2019 (7-class mapping) and SIIM-ISIC 2020 (binary: AUC and sensitivity at fixed specificity).
# TODO(2.19): pick one fixed unseen SIIM image, predict it and plot the 7-class bar chart plus P(malignant).
# TODO(2.20): explain logits -> softmax -> probabilities, then plot the reliability diagram and report ECE.

## Artifacts, reproducibility and submission packaging

Required: one self-contained notebook and a verified results archive. Produces: weights, metrics JSON, all figures, results ZIP.

In [ ]:
# TODO: save selected weights, metrics JSON and every report figure with fixed file names.
# TODO: store seeds, versions and the split manifest inside the results archive.
# TODO: rerun this notebook top-to-bottom in a fresh Colab runtime before packaging.
# TODO: export the results ZIP and verify its contents.